In [2]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from Modules.bond_finder import brute_force_bond_finder

In [27]:
"""
def bond_closure(G, edges):

    
        Takes an edge set E and finds it's bond closure, relative to some graph G
    

    # Handle empty set case
    if not edges:
        return frozenset()

    # Find connected components of subgraph induced by edges
    ccs = nx.connected_components(G.edge_subgraph(edges))

    # Return bond-closed set of edges
    return frozenset(
        e for cc in ccs for e in G.subgraph(cc).edges()
    )
"""

"\ndef bond_closure(G, edges):\n\n\n        Takes an edge set E and finds it's bond closure, relative to some graph G\n\n\n    # Handle empty set case\n    if not edges:\n        return frozenset()\n\n    # Find connected components of subgraph induced by edges\n    ccs = nx.connected_components(G.edge_subgraph(edges))\n\n    # Return bond-closed set of edges\n    return frozenset(\n        e for cc in ccs for e in G.subgraph(cc).edges()\n    )\n"

In [28]:
from collections import defaultdict

class DSU:
    def __init__(self, nodes):
        self.parent = {x: x for x in nodes}
        self.rank = {x: 0 for x in nodes}

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def bond_closure(G, edges):
    """
    Fast bond closure:
    - builds connectivity from edge set using DSU
    - then returns all edges of G inside induced components
    """

    if not edges:
        return frozenset()

    # Precompute nodes only from relevant edges
    nodes = set()
    for u, v in edges:
        nodes.add(u)
        nodes.add(v)

    dsu = DSU(nodes)

    # 1. Build connectivity from selected edges
    for u, v in edges:
        dsu.union(u, v)

    # 2. Group nodes by component root
    comp = defaultdict(list)
    for n in nodes:
        comp[dsu.find(n)].append(n)

    # 3. Fast membership test for "same component"
    node_root = {n: dsu.find(n) for n in nodes}

    # 4. Scan full edge set once
    result = []
    for u, v in G.edges():
        if u in node_root and v in node_root:
            if node_root[u] == node_root[v]:
                result.append((u, v))

    return frozenset(result)

In [29]:
def A_oplus_i(A, i, G):

    """
        A plus i operation in the next-closure algorithm
    """

    edges = list(sorted(G.edges()))

    B = frozenset([j for j in edges[:i] if j in A])

    return frozenset(
        bond_closure(G, B | frozenset({edges[i]}))
    )

In [30]:
def icover_check(A, B, i, G):

    """
        Checks whether A is i-covered by B for the next-closure algorithm.
    """

    edges = list(sorted(G.edges()))
    sym_diff = frozenset((A - B) | (B - A))

    if not sym_diff:
        return False

    if edges[i] not in B:
        return False
    
    if edges[i] != min(sym_diff):
        return False
    
    return True

In [31]:
def one_step(A, G):

    """
        Performs one step of the next-closure algorithm
    """

    edges = list(sorted(G.edges()))

    for i in reversed(range(len(edges))):
        if edges[i] in A:
            continue
        candidate = A_oplus_i(A, i, G)
        verdict = icover_check(A, candidate, i, G)
        
        if verdict == True:
            return candidate
    
    return None

In [34]:
def next_closure_algo(G):

    """
        Performs the next-closure algorithm to find all of the bonds of a graph
    """

    A = frozenset()
    bonds = [A]

    while True:
        A = one_step(A, G)

        if A is None:
            break

        bonds.append(A)

    return bonds